# Rationale Evaluation — Scoring LLM Reasoning Quality

Most LLM benchmarks only check whether a model reached the **correct answer**. That misses a critical failure mode: a model can guess "eligible" for entirely wrong reasons and still score 100%.

CivBench's differentiating capability is evaluating **how** a model reasons, not just what it concludes. Every `TestCase` ships with a `RationaleTrace` — a ground-truth, step-by-step policy reasoning chain. The `RationaleEvaluator` scores a model's free-text output against that trace across four dimensions:

| Dimension | Weight | What it measures |
|---|---|---|
| `step_coverage` | 35% | Did the model cover every key reasoning step? |
| `rule_accuracy` | 30% | Did it cite the correct CFR / policy rules? |
| `conclusion_correct` | 35% | Did it reach the correct final outcome? |
| `overall` | — | Weighted composite of the three above |

This notebook walks through each scoring dimension with concrete examples.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from govsynth import Pipeline
from govsynth.evaluation import RationaleEvaluator
from govsynth.models.rationale import RationaleTrace, ReasoningStep, PolicyCitation

print('govsynth imported successfully')

## 1. The RationaleTrace

A `RationaleTrace` documents every rule invoked, every threshold compared, and every deduction applied along the correct policy path. It is the ground truth that `RationaleEvaluator` scores against.

Generate one SNAP case and inspect its trace.

In [ ]:
pipeline = Pipeline.from_preset('snap.va')
cases = pipeline.generate(n=5, seed=42)

# Find a multi-step eligible case for a richer trace
eligible_cases = [c for c in cases if c.expected_outcome == 'eligible']
case = eligible_cases[0] if eligible_cases else cases[0]

print(f'Case ID:  {case.case_id}')
print(f'Outcome:  {case.expected_outcome.upper()}')
print(f'HH size:  {case.scenario.household_size}')
print(f'Gross:    ${case.scenario.monthly_gross_income:,.2f}/month')
if case.scenario.monthly_net_income is not None:
    print(f'Net:      ${case.scenario.monthly_net_income:,.2f}/month')
print(f'Assets:   ${case.scenario.liquid_assets:,.2f}')
print()

In [ ]:
# Inspect the full rationale trace in readable plain text
trace = case.rationale_trace
print(f'Trace has {trace.step_count()} steps')
print(f'Rules cited across all steps: {trace.cited_rules()}')
print(f'Determinative steps: {[s.title for s in trace.determinative_steps()]}')
print()
print('=== GROUND-TRUTH RATIONALE TRACE ===')
print(trace.to_plain_text())

### Trace structure

Each `ReasoningStep` has five fields:

- **`title`** — short label (e.g. `"Check gross income limit"`)
- **`rule_applied`** — the exact CFR section or policy rule (e.g. `"7 CFR 273.9(a)(1)"`)
- **`computation`** — what was compared or calculated in plain language
- **`result`** — outcome of this step (`"PASS"`, `"FAIL"`, or a derived value like `"net_income = $1,651"`)
- **`is_determinative`** — if `True`, this step alone decides the final outcome

The `RationaleEvaluator` uses `computation` to extract numeric signals (dollar amounts, percentages) and key terms, then checks whether those signals appear in the model's output.

## 2. Introduction to RationaleEvaluator

The evaluator is initialized with weights that must sum to 1.0. The defaults reflect the policy reasoning priorities of CivBench: step coverage and conclusion are equally weighted (35% each), with rule accuracy slightly lower (30%) to account for paraphrasing.

In [ ]:
evaluator = RationaleEvaluator(
    step_weight=0.35,
    rule_weight=0.30,
    conclusion_weight=0.35,
)

print('Evaluator weights:')
print(f'  step_coverage:      {evaluator.step_weight:.0%}')
print(f'  rule_accuracy:      {evaluator.rule_weight:.0%}')
print(f'  conclusion_correct: {evaluator.conclusion_weight:.0%}')

## 3. Scoring a Perfect Response

Craft a model output that hits every reasoning step, cites the correct CFR sections, and states the correct conclusion. This is what a well-calibrated LLM should produce.

In [ ]:
# Build a perfect response that mirrors the trace exactly
hh = case.scenario.household_size
gross = case.scenario.monthly_gross_income
net = case.scenario.monthly_net_income or (gross * 0.85)
assets = case.scenario.liquid_assets
outcome = case.expected_outcome

# Extract the key numbers from the trace to mirror back
step_computations = [s.computation for s in trace.steps]
rules = trace.cited_rules()

perfect_response = (
    f"I will analyze this household's SNAP eligibility step by step, "
    f"citing the applicable federal regulations.\n\n"
    f"Step 1: Check gross income limit (7 CFR 273.9(a)(1))\n"
    f"{step_computations[0]}\n\n"
)
if len(step_computations) > 1:
    perfect_response += (
        f"Step 2: {trace.steps[1].title} ({trace.steps[1].rule_applied})\n"
        f"{step_computations[1]}\n\n"
    )
if len(step_computations) > 2:
    for i, step in enumerate(trace.steps[2:], start=3):
        perfect_response += (
            f"Step {i}: {step.title} ({step.rule_applied})\n"
            f"{step.computation}\n\n"
        )
perfect_response += f"Conclusion: {trace.conclusion}\n"
perfect_response += f"This household is {outcome} for SNAP."

print('--- PERFECT MODEL OUTPUT ---')
print(perfect_response)
print()

In [ ]:
score_perfect = evaluator.score(case, perfect_response)
print(score_perfect)
print()
print(f'Steps covered: {score_perfect.steps_covered}')
print(f'Steps missed:  {score_perfect.steps_missed}')
print(f'Rules cited:   {score_perfect.rules_cited}')
print(f'Rules missing: {score_perfect.rules_missing}')

## 4. Scoring a Partial Response

A common LLM failure: the model reaches the **right answer** but skips intermediate steps — for example, jumping straight from gross income to a conclusion without showing the net income calculation.

This inflates `conclusion_correct` while `step_coverage` drops, exposing the gap.

In [ ]:
# Partial response: mentions the correct outcome and cites one rule,
# but skips the net income calculation step entirely
first_step = trace.steps[0]

partial_response = (
    f"Looking at this case, I can see the household has {hh} member(s). "
    f"Under 7 CFR 273.9(a)(1), I need to check the gross income limit. "
    f"{first_step.computation} "
    f"Therefore, the household is {outcome} for SNAP."
)

print('--- PARTIAL MODEL OUTPUT (skips net income step) ---')
print(partial_response)
print()

In [ ]:
score_partial = evaluator.score(case, partial_response)
print(score_partial)
print()
print(f'Steps covered: {score_partial.steps_covered}')
print(f'Steps missed:  {score_partial.steps_missed}')
print()
print('Compare step_coverage: perfect vs partial')
print(f'  Perfect: {score_perfect.step_coverage:.2f}')
print(f'  Partial: {score_partial.step_coverage:.2f}  ← dropped due to skipped steps')
print()
print('Overall score impact:')
print(f'  Perfect: {score_perfect.overall:.2f}')
print(f'  Partial: {score_partial.overall:.2f}')

## 5. Scoring a Wrong-Answer Response

A model that concludes the opposite of the correct outcome — for example, saying "eligible" when the answer is "ineligible" — receives a `conclusion_correct` of 0.0. This tanks the overall score even if the model cited the right rules and followed the right steps, which is the intended behavior: an incorrect conclusion in a benefits determination can cause real harm.

In [ ]:
# Find an ineligible case for a clearer wrong-answer demonstration
ineligible_cases = [c for c in cases if c.expected_outcome == 'ineligible']
ineligible_case = ineligible_cases[0] if ineligible_cases else cases[-1]

print(f'Case: {ineligible_case.case_id}')
print(f'Correct outcome: {ineligible_case.expected_outcome.upper()}')
print()
print(ineligible_case.rationale_trace.to_plain_text())

In [ ]:
# Wrong-answer response: cites the right rule and shows correct computations,
# but flips the conclusion to 'eligible'
ic = ineligible_case
ic_trace = ic.rationale_trace
ic_step0 = ic_trace.steps[0]

wrong_answer_response = (
    f"Analyzing SNAP eligibility for this {ic.scenario.household_size}-person household.\n"
    f"Per 7 CFR 273.9(a)(1): {ic_step0.computation}\n"
    f"After reviewing all applicable deductions under 7 CFR 273.9(c), "
    f"this household qualifies and is eligible for SNAP benefits."
    # Note: cites rules correctly but flips the conclusion
)

print('--- WRONG-ANSWER MODEL OUTPUT ---')
print(wrong_answer_response)
print()

In [ ]:
score_wrong = evaluator.score(ineligible_case, wrong_answer_response)
print(score_wrong)
print()
print('Score breakdown:')
print(f'  step_coverage:      {score_wrong.step_coverage:.2f}  (model showed computations)')
print(f'  rule_accuracy:      {score_wrong.rule_accuracy:.2f}  (cited correct CFR sections)')
print(f'  conclusion_correct: {score_wrong.conclusion_correct:.2f}  ← 0.0 because outcome is WRONG')
print(f'  overall:            {score_wrong.overall:.2f}')
print()
print(f'  predicted: "{score_wrong.predicted_outcome}"  expected: "{score_wrong.expected_outcome}"')
print()
print('A wrong conclusion fails the test even with correct reasoning steps.')

## 6. Scoring a Rule-Hallucinating Response

LLMs frequently cite plausible-looking but fabricated CFR sections. The evaluator checks whether each ground-truth rule's CFR part number (e.g. `273.9`) appears in the model output. A model that invents `7 CFR 999.1` or `7 CFR 123.4` while omitting the real citations gets penalized on `rule_accuracy`.

In [ ]:
# Hallucinating response: correct conclusion, but cites fake CFR sections
outcome_word = case.expected_outcome
hh_step0 = trace.steps[0]

hallucinating_response = (
    f"This household has {hh} members with a gross monthly income of ${gross:,.2f}. "
    f"Under 7 CFR 999.1 (SNAP income screening), households must meet the gross income test. "
    f"{hh_step0.computation} "
    f"After applying deductions per 7 CFR 123.4(b) (standard deduction schedule), "
    f"net income is ${net:,.2f}. "
    f"Per 7 CFR 456.7, assets of ${assets:,.2f} are within allowable limits. "
    f"Conclusion: this household is {outcome_word} for SNAP."
)

print('--- HALLUCINATING MODEL OUTPUT (fake CFR sections) ---')
print(hallucinating_response)
print()
print(f'Ground-truth rules: {trace.cited_rules()}')
print('Fake rules cited: ["7 CFR 999.1", "7 CFR 123.4(b)", "7 CFR 456.7"]')

In [ ]:
score_halluc = evaluator.score(case, hallucinating_response)
print(score_halluc)
print()
print('Score breakdown:')
print(f'  step_coverage:      {score_halluc.step_coverage:.2f}  (computation values matched)')
print(f'  rule_accuracy:      {score_halluc.rule_accuracy:.2f}  ← penalized for hallucinated CFR sections')
print(f'  conclusion_correct: {score_halluc.conclusion_correct:.2f}  (outcome is correct)')
print(f'  overall:            {score_halluc.overall:.2f}')
print()
print(f'  Rules cited (from ground truth): {score_halluc.rules_cited}')
print(f'  Rules missing:                   {score_halluc.rules_missing}')
print()
print('Rule accuracy isolates citation quality independent of conclusion accuracy.')

## 7. Score Breakdown Table

Generate 5 cases and score three simulated response types per case — a good response, a partial response, and a wrong-answer response. This shows how the three dimensions move independently.

In [ ]:
all_cases = pipeline.generate(n=5, seed=99)

def make_good_response(c):
    """Mirrors the trace: all steps, correct rules, correct conclusion."""
    t = c.rationale_trace
    parts = []
    for step in t.steps:
        parts.append(f"{step.title} ({step.rule_applied}): {step.computation}")
    parts.append(f"Conclusion: {t.conclusion}")
    parts.append(f"This household is {c.expected_outcome} for SNAP.")
    return " ".join(parts)

def make_partial_response(c):
    """Only covers the first step, skips the rest."""
    t = c.rationale_trace
    s0 = t.steps[0]
    return (
        f"{s0.title} ({s0.rule_applied}): {s0.computation} "
        f"This household is {c.expected_outcome} for SNAP."
    )

def make_wrong_response(c):
    """Correct steps but wrong conclusion (flipped outcome)."""
    t = c.rationale_trace
    flipped = 'ineligible' if c.expected_outcome == 'eligible' else 'eligible'
    parts = []
    for step in t.steps:
        parts.append(f"{step.title} ({step.rule_applied}): {step.computation}")
    parts.append(f"This household is {flipped} for SNAP.")
    return " ".join(parts)

print(f'Scoring 3 response types across {len(all_cases)} cases...')

In [ ]:
rows = []
for c in all_cases:
    for label, fn in [('good', make_good_response), ('partial', make_partial_response), ('wrong', make_wrong_response)]:
        s = evaluator.score(c, fn(c))
        rows.append({
            'case_id': c.case_id.split('.')[-2],   # short label
            'expected': c.expected_outcome,
            'response_type': label,
            'step_cov': s.step_coverage,
            'rule_acc': s.rule_accuracy,
            'conclusion': s.conclusion_correct,
            'overall': s.overall,
            'pass': s.passed(),
        })

# Print as formatted table
header = f"{'Case':<22} {'Expected':<12} {'Type':<10} {'StepCov':>8} {'RuleAcc':>8} {'Concl':>7} {'Overall':>8} {'Pass':>5}"
print(header)
print('-' * len(header))

for row in rows:
    pass_str = 'PASS' if row['pass'] else 'FAIL'
    print(
        f"{row['case_id']:<22} {row['expected']:<12} {row['response_type']:<10} "
        f"{row['step_cov']:>8.2f} {row['rule_acc']:>8.2f} {row['conclusion']:>7.2f} "
        f"{row['overall']:>8.2f} {pass_str:>5}"
    )

In [ ]:
# Aggregate summary stats by response type
from collections import defaultdict

by_type = defaultdict(list)
for row in rows:
    by_type[row['response_type']].append(row)

print(f"\n{'Response Type':<12} {'Avg StepCov':>12} {'Avg RuleAcc':>12} {'Avg Concl':>10} {'Avg Overall':>12} {'Pass Rate':>10}")
print('-' * 70)
for rtype in ['good', 'partial', 'wrong']:
    group = by_type[rtype]
    n = len(group)
    avg_step = sum(r['step_cov'] for r in group) / n
    avg_rule = sum(r['rule_acc'] for r in group) / n
    avg_concl = sum(r['conclusion'] for r in group) / n
    avg_overall = sum(r['overall'] for r in group) / n
    pass_rate = sum(1 for r in group if r['pass']) / n
    print(f"{rtype:<12} {avg_step:>12.2f} {avg_rule:>12.2f} {avg_concl:>10.2f} {avg_overall:>12.2f} {pass_rate:>9.0%}")

## 8. Using score_batch and summary_stats

For real evaluation runs you will score many (case, output) pairs at once. `score_batch` and `summary_stats` handle this efficiently.

In [ ]:
# Score a batch of 'good' responses and get aggregate stats
good_outputs = [make_good_response(c) for c in all_cases]
batch_scores = evaluator.score_batch(all_cases, good_outputs)

stats = evaluator.summary_stats(batch_scores)
print('summary_stats for perfect responses:')
for k, v in stats.items():
    if isinstance(v, float):
        print(f'  {k:<30} {v:.3f}')
    else:
        print(f'  {k:<30} {v}')

# Now compare with partial responses
partial_outputs = [make_partial_response(c) for c in all_cases]
partial_scores = evaluator.score_batch(all_cases, partial_outputs)
partial_stats = evaluator.summary_stats(partial_scores)

print()
print('summary_stats for partial responses:')
for k, v in partial_stats.items():
    if isinstance(v, float):
        print(f'  {k:<30} {v:.3f}')
    else:
        print(f'  {k:<30} {v}')

## 9. Custom Evaluator Weights

The default weights (35 / 30 / 35) are suitable for eligibility determination tasks. For **policy QA** tasks where citation accuracy is paramount, you might raise `rule_weight`. For **agentic** tasks where reaching the correct outcome is the primary metric, raise `conclusion_weight`.

Weights must sum to exactly 1.0.

In [ ]:
# Citation-heavy evaluator: rule accuracy matters most
citation_evaluator = RationaleEvaluator(
    step_weight=0.25,
    rule_weight=0.50,  # citations are primary
    conclusion_weight=0.25,
)

# Outcome-first evaluator: conclusion is the primary signal
outcome_evaluator = RationaleEvaluator(
    step_weight=0.20,
    rule_weight=0.20,
    conclusion_weight=0.60,  # outcome is primary
)

# Score the hallucinating response (correct conclusion, wrong CFR) under each evaluator
score_default = evaluator.score(case, hallucinating_response)
score_citation = citation_evaluator.score(case, hallucinating_response)
score_outcome = outcome_evaluator.score(case, hallucinating_response)

print('Hallucinating response (correct outcome, fake CFR) under different evaluators:')
print(f"{'Evaluator':<25} {'StepCov':>8} {'RuleAcc':>8} {'Concl':>7} {'Overall':>8}")
print('-' * 60)
for label, s in [('default (35/30/35)', score_default), ('citation (25/50/25)', score_citation), ('outcome (20/20/60)', score_outcome)]:
    print(f"{label:<25} {s.step_coverage:>8.2f} {s.rule_accuracy:>8.2f} {s.conclusion_correct:>7.2f} {s.overall:>8.2f}")

print()
print('The citation-heavy evaluator penalizes hallucinated CFR sections most severely.')
print('The outcome-first evaluator rewards the correct conclusion despite bad citations.')

## 10. What Makes a Good vs. Bad Rationale

### Good rationale (high score)
- **Covers every step**: addresses gross income check, deduction calculations, net income check, and asset test (where applicable)
- **Cites exact CFR sections**: includes the specific part numbers from the ground-truth trace (e.g. `7 CFR 273.9(a)(1)`, `7 CFR 273.9(c)`)
- **Uses the correct numeric values**: dollar amounts from the case scenario appear in the output
- **States a clear, unambiguous conclusion**: `"eligible"` or `"ineligible"` — no hedging that produces an `"ambiguous"` predicted outcome

### Bad rationale (low score)

| Failure mode | Dimension penalized | Description |
|---|---|---|
| Skips steps | `step_coverage` | Jumps from gross income check to conclusion without net income calculation |
| Hallucinated citations | `rule_accuracy` | Cites `7 CFR 999.1` or other fabricated CFR sections |
| Wrong conclusion | `conclusion_correct` | Says "eligible" when the correct answer is "ineligible" |
| Hedged conclusion | `conclusion_correct` | Says "likely eligible" — triggers ambiguous detection (0.5) |
| Omits dollar amounts | `step_coverage` | Doesn't include the threshold values used in comparisons |

### The passing threshold

`RationaleScore.passed(threshold=0.7)` returns `True` if `overall >= 0.7`. This default reflects a practical floor: a model scoring below 0.7 is either missing too many reasoning steps, citing wrong rules, or drawing the wrong conclusion — any of which is unacceptable in a government benefits context.

Raise the threshold to `0.85` or `0.9` for high-stakes evaluation scenarios where policy accuracy is critical.

In [ ]:
# Demonstrate the passing threshold at different levels
print('Passing threshold comparison across response types (default case):')
print(f"{'Response':<12} {'Overall':>8} {'pass(0.7)':>10} {'pass(0.85)':>11} {'pass(0.9)':>10}")
print('-' * 55)
for label, s in [('perfect', score_perfect), ('partial', score_partial), ('hallucinating', score_halluc)]:
    p70 = s.passed(0.7)
    p85 = s.passed(0.85)
    p90 = s.passed(0.9)
    print(f"{label:<12} {s.overall:>8.2f} {'PASS' if p70 else 'FAIL':>10} {'PASS' if p85 else 'FAIL':>11} {'PASS' if p90 else 'FAIL':>10}")

## Next Steps

- `notebooks/01_quickstart.ipynb` — library overview and first test case generation
- `notebooks/02_snap_edge_cases.ipynb` — generating edge cases at policy thresholds
- `notebooks/03_wic_eligibility.ipynb` — WIC eligibility case generation
- `notebooks/04_batch_pipeline.ipynb` — batch generation across jurisdictions
- `govsynth/evaluation/rationale_evaluator.py` — full `RationaleEvaluator` source
- `govsynth/models/rationale.py` — `RationaleTrace` and `ReasoningStep` data models